# Activity classification — four-condition evaluation

Tests whether sampling choices change X-CLIP zero-shot activity rankings at a face detection: `consecutive_32` (the frames [f-16, f+15], clamped at the video edges) versus sparse uniform temporal windows of 5s, 10s, and 15s (32 samples each). All reference labels come from the manual `REFERENCE_LABELS` mapping, never from filenames or model output. The notebook is strictly read-only with respect to the analysis database.

In [ ]:
DB_PATH = None
DETECTION_IDS = None  # None auto-selects one usable detection at random.
SEED = 41  # None = fresh random seed each run; set an int to reproduce.
ACTIVITIES = [
    "Dancing", "Rapping", "Singing", "Playing sports", "Playing games",
    "Running", "Walking", "Sitting and listening", "Drawing or painting",
]
DESCRIPTIONS = None
DEVICE = "auto"
MODEL_PATH = None
CACHE_DIR = None
REFERENCE_LABELS = {}  # detection_id -> exact label from ACTIVITIES
MAX_VIDEO_SECONDS = 120.0
TOP_K = 3


In [ ]:
import sys
from contextlib import closing
from dataclasses import asdict
from pathlib import Path
from time import perf_counter

_here = Path.cwd()
if (_here / "analysis_db").is_dir():
    backend_dir = _here
elif _here.name == "notebooks" and (_here.parent / "analysis_db").is_dir():
    backend_dir = _here.parent
else:
    raise RuntimeError(
        "Launch this notebook from backend/ or backend/notebooks/. The current"
        f" working directory ({_here}) is not a backend directory with an"
        " analysis_db/ package and is not backend/notebooks/. Open the notebook"
        " through backend/notebooks/ or run Jupyter with the working directory"
        " set to backend/.")
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

import av
import matplotlib.pyplot as plt
import numpy as np
import random
from IPython.display import Markdown, display

from activity_classification import XCLIPClassifier
from analysis_db.paths import DEFAULT_DB, REPOSITORY_ROOT
from analysis_db.storage import connect_database
from video_retrieval import VideoRetriever


In [ ]:
for reference_detection, reference_label in REFERENCE_LABELS.items():
    if reference_label not in ACTIVITIES:
        raise RuntimeError(
            f"Invalid configuration: REFERENCE_LABELS[{reference_detection!r}] ="
            f" {reference_label!r} is not in ACTIVITIES {ACTIVITIES}."
            " Set the intended label exactly as it appears in ACTIVITIES, or"
            " delete the entry to leave the detection unlabeled.")

db_path = Path(DEFAULT_DB if DB_PATH is None else DB_PATH).resolve()
if not db_path.is_file():
    raise RuntimeError(
        f"Database not found: {db_path}. Initialize and ingest a database first"
        " (analysis_db.storage.initialize_database + the ingest script) or set"
        " DB_PATH to an existing analysis.sqlite.")
retriever = VideoRetriever(db_path)

DETECTION_QUERY = (
    "SELECT v.video_id, v.title, v.filename, v.file_path, v.duration_seconds,"
    " v.fps, v.width, v.height, d.detection_id, d.person_id, f.frame_id,"
    " f.frame_number, f.source_time_seconds, d.x1, d.y1, d.x2, d.y2"
    " FROM Video AS v"
    " JOIN Frame AS f ON f.video_id = v.video_id"
    " JOIN FaceDetection AS d ON d.frame_id = f.frame_id")

def _detection_record(row):
    return {
        "video_id": row[0],
        "video_title": row[1],
        "video_filename": row[2],
        "video_path": (REPOSITORY_ROOT / str(row[3])).resolve(),
        "duration_seconds": row[4],
        "fps": row[5],
        "width": row[6],
        "height": row[7],
        "detection_id": row[8],
        "person_id": row[9],
        "frame_id": row[10],
        "frame_number": row[11],
        "source_time_seconds": row[12],
        "bbox": (float(row[13]), float(row[14]), float(row[15]), float(row[16])),
    }

def _usable_file(file_path):
    if file_path is None:
        return False
    return (REPOSITORY_ROOT / str(file_path)).resolve().is_file()

if SEED is not None and (
        isinstance(SEED, bool) or not isinstance(SEED, int) or SEED < 0):
    raise RuntimeError(
        f"Invalid configuration: SEED = {SEED!r} must be a nonnegative"
        " integer or None (None draws a fresh random seed for the run and"
        " reports it).")
selection_seed = SEED if SEED is not None else random.getrandbits(64)
selection_rng = random.Random(selection_seed)

if DETECTION_IDS is None:
    with closing(connect_database(db_path, read_only=True)) as connection:
        auto_rows = connection.execute(
            DETECTION_QUERY
            + " WHERE v.duration_seconds IS NOT NULL AND v.duration_seconds > 0"
            + " AND v.duration_seconds <= ?"
            + " ORDER BY v.video_id ASC, d.detection_id ASC",
            (float(MAX_VIDEO_SECONDS),),
        ).fetchall()
    candidates = [
        _detection_record(row) for row in auto_rows if _usable_file(row[3])]
    selection_rng.shuffle(candidates)
    selected = []
    for candidate in candidates:
        try:
            retriever.get_clip(candidate["detection_id"])
        except Exception as exc:
            display(Markdown(
                f"- skipping detection {candidate['detection_id']} in"
                f" {candidate['video_filename']!r}: {type(exc).__name__}"))
            continue
        selected.append(candidate)
        break
    if not selected:
        raise RuntimeError(
            f"No evaluable detection found in {db_path}. Auto-selection"
            " needs a Video row with a positive duration_seconds no larger"
            " than MAX_VIDEO_SECONDS, an existing original file, and a"
            " decodable detection. Ingest data first or set DB_PATH and"
            " DETECTION_IDS explicitly.")
    display(Markdown(
        f"- Auto-selection used seed `{selection_seed}` over"
        f" {len(candidates)} eligible detection(s); picked"
        f" detection {selected[0]['detection_id']}."))
else:
    selected = []
    for requested_id in DETECTION_IDS:
        with closing(connect_database(db_path, read_only=True)) as connection:
            row = connection.execute(
                DETECTION_QUERY + " WHERE d.detection_id = ?",
                (requested_id,),
            ).fetchone()
        if row is None:
            raise RuntimeError(
                f"DETECTION_IDS contains unknown detection {requested_id!r} for"
                f" database {db_path}. Query FaceDetection for valid ids or"
                " remove the entry.")
        if not _usable_file(row[3]):
            display(Markdown(
                f"- detection {requested_id}: original file {row[3]!r} is"
                " missing on disk; it stays enlisted and its conditions will be"
                " recorded as failed."))
        selected.append(_detection_record(row))
    if not selected:
        raise RuntimeError("DETECTION_IDS is empty; nothing to evaluate.")


In [ ]:
VIDEO_PTS_SECONDS = {}

def decoded_frame_seconds(video_path):
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        time_base = stream.time_base
        stamps = []
        for frame in container.decode(video=0):
            if frame.pts is None or time_base is None or time_base <= 0:
                raise RuntimeError(
                    f"{video_path}: decoded frame has no usable PTS/time base")
            stamps.append(float(frame.pts * time_base))
    if not stamps:
        raise RuntimeError(f"{video_path}: no video frames decoded")
    return stamps

for video_record in {record["video_id"]: record for record in selected}.values():
    VIDEO_PTS_SECONDS[video_record["video_id"]] = decoded_frame_seconds(
        video_record["video_path"])

def consecutive_indices(frame_number, last_index):
    window = frame_number + np.arange(32) - 16
    return tuple(np.clip(window, 0, max(last_index, 0)).tolist())

def consecutive_diagnostics(video_id, frame_number):
    stamps = VIDEO_PTS_SECONDS[video_id]
    last_index = len(stamps) - 1
    indices = consecutive_indices(frame_number, last_index)
    unique = sorted(set(indices))
    return {
        "source_span_seconds": stamps[unique[-1]] - stamps[unique[0]],
        "unique_frame_count": len(unique),
        "left_padding_count": max(0, 16 - frame_number),
        "right_padding_count": max(0, frame_number + 15 - last_index),
        "reference_pts_seconds": stamps[frame_number]
        if frame_number <= last_index else None,
        "start_pts_seconds": stamps[unique[0]],
        "end_pts_seconds": stamps[unique[-1]],
        "last_decoded_index": last_index,
    }

def small_preview(frame, step=4):
    return np.ascontiguousarray(frame[::step, ::step])

initial_detection_id = selected[0]["detection_id"]
initial_retrieval_seconds = None
first_clip = None
try:
    clip_start = perf_counter()
    first_clip = retriever.get_clip(initial_detection_id)
    initial_retrieval_seconds = perf_counter() - clip_start
except Exception as exc:
    display(Markdown(
        f"- preview failed for detection {initial_detection_id}:"
        f" {type(exc).__name__}; it stays enlisted and all of its conditions"
        " will be recorded as failed."))

metadata_lines = []
for record in selected:
    duration = record["duration_seconds"]
    duration_text = f"{duration:.1f}s" if duration is not None else "unknown"
    stored = record["source_time_seconds"]
    stored_text = f"{stored:.3f}s" if stored is not None else "unreported"
    stamps = VIDEO_PTS_SECONDS.get(record["video_id"], [])
    decoded = (
        stamps[record["frame_number"]]
        if record["frame_number"] < len(stamps) else None)
    drift_text = ""
    if stored is not None and decoded is not None and abs(stored - decoded) > 0.05:
        drift_text = (
            f" NOTE: stored time disagrees with decoded PTS ({decoded:.3f}s).")
    metadata_lines.append(
        f"- detection `{record['detection_id']}` person `{record['person_id']}`"
        f" in video {record['video_id']} ({record['video_title']!r},"
        f" {record['video_filename']!r} {record['width']}x{record['height']},"
        f" {duration_text}): frame {record['frame_number']} at stored"
        f" {stored_text}, bbox ({record['bbox']},)".rstrip(", ") + drift_text)
display(Markdown("## Selected detections\n" + "\n".join(metadata_lines)))

condition_previews = {}
if first_clip is not None:
    condition_previews[(initial_detection_id, "consecutive_32")] = [
        small_preview(first_clip[slot]) for slot in (0, 16, 31)
    ]
    figure, axes = plt.subplots(1, 3, figsize=(9, 3))
    for axis, slot in zip(axes, (0, 16, 31)):
        axis.imshow(first_clip[slot])
        axis.set_title(f"slot {slot}")
        axis.axis("off")
    display(figure)
    plt.close(figure)
    del figure, axes


In [ ]:
UNREPORTED = "unreported until first classify"

def _shown(value, quoted):
    if value is None:
        return UNREPORTED
    return f"`{value}`" if quoted else str(value)

load_start = perf_counter()
classifier = XCLIPClassifier(device=DEVICE, model_path=MODEL_PATH, cache_dir=CACHE_DIR)
model_load_seconds = perf_counter() - load_start
display(Markdown(
    "## Classifier\n"
    f"- model: {_shown(getattr(classifier, 'model', None), quoted=True)},"
    f" revision: {_shown(getattr(classifier, 'model_revision', None), quoted=True)}\n"
    f"- device: {_shown(getattr(classifier, 'device', None), quoted=True)},"
    f" dtype: {_shown(getattr(classifier, 'dtype', None), quoted=True)}\n"
    f"- frames per classify: {_shown(getattr(classifier, 'num_frames', None), quoted=False)}\n"
    f"- load time: {model_load_seconds:.2f}s"))


In [ ]:
initial_result = None
if first_clip is not None:
    _ = classifier.classify(first_clip, activities=ACTIVITIES, descriptions=DESCRIPTIONS)
    classify_start = perf_counter()
    initial_result = classifier.classify(
        first_clip, activities=ACTIVITIES, descriptions=DESCRIPTIONS)
    initial_classify_seconds = perf_counter() - classify_start
    first_clip = None
    ranked_lines = [
        f"{index + 1}. **{prediction.label}**"
        f" (score {prediction.score:.3f}, logit {prediction.logit:.3f})"
        for index, prediction in enumerate(initial_result.predictions[:TOP_K])
    ]
    reference_label = REFERENCE_LABELS.get(initial_detection_id)
    if reference_label is None:
        placement = "No manual reference label for this detection."
    else:
        labels = [prediction.label for prediction in initial_result.predictions]
        placement = (
            f"Reference label `{reference_label}` ranks"
            f" {labels.index(reference_label) + 1} of {len(labels)}."
            if reference_label in labels
            else f"Reference label `{reference_label}` is not in the ranking.")
    display(Markdown(
        f"## Initial prediction (consecutive 32 frames)\n"
        + "\n".join(ranked_lines)
        + f"\n\n{placement}\n\nPreprocessing"
        f" {initial_result.preprocessing_time_seconds:.3f}s, inference"
        f" {initial_result.inference_time_seconds:.3f}s."))
else:
    display(Markdown(
        "No preview clip was available; the first detection stays recorded as"
        " failed and its conditions remain unmeasured."))


In [ ]:
CONDITIONS = (
    ("consecutive_32", None),
    ("uniform_5s", 5.0),
    ("uniform_10s", 10.0),
    ("uniform_15s", 15.0),
)
CONDITION_NAMES = tuple(name for name, _ in CONDITIONS)
comparison_rows = []
experiment_errors = {name: [] for name in CONDITION_NAMES}
rows_by_key = {}

def _clip_frames(clip):
    if hasattr(clip, "shape"):
        return clip
    return clip.frames

for detection in selected:
    detection_id = detection["detection_id"]
    reference_label = REFERENCE_LABELS.get(detection_id)
    for condition_name, interval_seconds in CONDITIONS:
        reuse_initial = (
            condition_name == "consecutive_32"
            and detection_id == initial_detection_id
            and initial_result is not None
        )
        clip = None
        result = None
        try:
            if reuse_initial:
                result = initial_result
                retrieval_seconds = initial_retrieval_seconds
            else:
                retrieval_start = perf_counter()
                if interval_seconds is None:
                    clip = retriever.get_clip(detection_id)
                else:
                    clip = retriever.get_temporal_clip(
                        detection_id,
                        interval_seconds=interval_seconds,
                        sample_count=32,
                    )
                retrieval_seconds = perf_counter() - retrieval_start
                frames = _clip_frames(clip)
                condition_previews[(detection_id, condition_name)] = [
                    small_preview(frames[slot]) for slot in (0, 16, 31)
                ]
                result = classifier.classify(
                    frames, activities=ACTIVITIES, descriptions=DESCRIPTIONS)
        except Exception as exc:
            step = "inference" if clip is not None else "retrieval"
            experiment_errors[condition_name].append({
                "detection_id": detection_id,
                "step": step,
                "error": type(exc).__name__,
            })
            condition_previews.pop((detection_id, condition_name), None)
            clip = result = None
            continue
        if interval_seconds is None:
            diagnostics = consecutive_diagnostics(
                detection["video_id"], detection["frame_number"])
            unique_frame_count = diagnostics["unique_frame_count"]
            left_padding_count = diagnostics["left_padding_count"]
            right_padding_count = diagnostics["right_padding_count"]
            source_span_seconds = diagnostics["source_span_seconds"]
        else:
            timestamps = clip.source_timestamps_seconds
            unique_frame_count = clip.unique_frame_count
            left_padding_count = clip.left_padding_count
            right_padding_count = clip.right_padding_count
            source_span_seconds = (
                float(max(timestamps) - min(timestamps)) if timestamps else None
            )
        prediction_labels = [
            prediction.label for prediction in result.predictions
        ]
        if reference_label is None:
            reference_rank = None
            reference_score = None
            top1_correct = None
            top3_correct = None
        else:
            if reference_label in prediction_labels:
                reference_rank = prediction_labels.index(reference_label) + 1
                reference_score = result.predictions[reference_rank - 1].score
            else:
                reference_rank = None
                reference_score = None
            top1_correct = prediction_labels[0] == reference_label
            top3_correct = reference_label in prediction_labels[:TOP_K]
        row = {
            "detection_id": detection_id,
            "condition": condition_name,
            "num_frames": result.num_frames,
            "predictions": [asdict(prediction) for prediction in result.predictions],
            "reference_label": reference_label,
            "reference_rank": reference_rank,
            "reference_score": reference_score,
            "top1_correct": top1_correct,
            "top3_correct": top3_correct,
            "retrieval_time_seconds": retrieval_seconds,
            "preprocessing_time_seconds": result.preprocessing_time_seconds,
            "inference_time_seconds": result.inference_time_seconds,
            "source_span_seconds": source_span_seconds,
            "unique_frame_count": unique_frame_count,
            "left_padding_count": left_padding_count,
            "right_padding_count": right_padding_count,
        }
        comparison_rows.append(row)
        rows_by_key[(detection_id, condition_name)] = row
        clip = result = None


In [ ]:
rows_by_condition = {
    name: [
        row["detection_id"] for row in comparison_rows
        if row["condition"] == name
    ]
    for name in CONDITION_NAMES
}
failed_by_condition = {
    name: sorted({record["detection_id"] for record in experiment_errors[name]})
    for name in CONDITION_NAMES
}
all_selected_ids = [record["detection_id"] for record in selected]
labeled_ids = [
    detection_id for detection_id in all_selected_ids
    if detection_id in REFERENCE_LABELS
]
unlabeled_ids = [
    detection_id for detection_id in all_selected_ids
    if detection_id not in REFERENCE_LABELS
]
common_successful = [
    detection_id for detection_id in labeled_ids
    if all(detection_id in rows_by_condition[name] for name in CONDITION_NAMES)
]

def _accuracy(condition_name, key):
    values = [
        rows_by_key[(detection_id, condition_name)][key]
        for detection_id in common_successful
    ]
    return sum(1 for value in values if value) / len(values)

if common_successful:
    denominator = (
        f"{len(common_successful)} labeled detection(s) succeeded in all four"
        f" conditions ({', '.join(str(d) for d in common_successful)});"
        " accuracies share this denominator.")
else:
    denominator = (
        "Accuracy was not measured: no labeled detection succeeded in all four"
        " conditions; only rankings are reported.")
quality_summary = []
for name in CONDITION_NAMES:
    failed_here = set(failed_by_condition[name])
    failed_elsewhere = sorted(set().union(
        *[failed_by_condition[other] for other in CONDITION_NAMES
          if other != name]) - failed_here)
    quality_summary.append({
        "condition": name,
        "evaluated_detections": len(common_successful),
        "top1_accuracy": _accuracy(name, "top1_correct")
        if common_successful else None,
        "top3_accuracy": _accuracy(name, "top3_correct")
        if common_successful else None,
        "excluded": {
            "failed_in_this_condition": sorted(failed_here),
            "failed_in_other_conditions": failed_elsewhere,
            "unlabeled": unlabeled_ids,
        },
    })

def _seconds(value):
    return "-" if value is None else f"{value:.3f}"

summary_lines = [
    "## Quality summary\n\n" + denominator + "\n",
    "| condition | evaluated | top-1 | top-3 | excluded: failed here, failed elsewhere, unlabeled |",
    "|---|---|---|---|---|",
]
for summary in quality_summary:
    top1 = summary["top1_accuracy"]
    top3 = summary["top3_accuracy"]
    excluded = summary["excluded"]
    summary_lines.append(
        f"| {summary['condition']} | {summary['evaluated_detections']} |"
        f" {'n/a' if top1 is None else format(top1, '.2f')} |"
        f" {'n/a' if top3 is None else format(top3, '.2f')} |"
        f" {excluded['failed_in_this_condition']},"
        f" {excluded['failed_in_other_conditions']},"
        f" {excluded['unlabeled']} |")
rows_lines = [
    "### Per-row comparison (quality and labels)\n",
    "| condition | detection | reference | rank | top-1 label | top-1 | top-3 |",
    "|---|---|---|---|---|---|---|",
]
sampling_lines = [
    "\n### Per-row comparison (sampling and timing)\n",
    "| condition | detection | num_frames | retrieval (s) | preprocessing (s) | inference (s) | span (s) | unique | pad L/R |",
    "|---|---|---|---|---|---|---|---|---|",
]
for row in comparison_rows:
    top1_label = (
        row["predictions"][0]["label"] if row["predictions"] else "-")
    rows_lines.append(
        f"| {row['condition']} | {row['detection_id']} |"
        f" {row['reference_label']} |"
        f" {row['reference_rank'] if row['reference_rank'] is not None else '-'} |"
        f" {top1_label} | {row['top1_correct']} | {row['top3_correct']} |")
    sampling_lines.append(
        f"| {row['condition']} | {row['detection_id']} | {row['num_frames']} |"
        f" {_seconds(row['retrieval_time_seconds'])} |"
        f" {_seconds(row['preprocessing_time_seconds'])} |"
        f" {_seconds(row['inference_time_seconds'])} |"
        f" {_seconds(row['source_span_seconds'])} |"
        f" {row['unique_frame_count']} |"
        f" {row['left_padding_count']}/{row['right_padding_count']} |")
notes_text = (
    "\n#### Interpretation caveats\n\n"
    "- Scene cuts and unrelated people or scenery inside a window can dominate"
    " the visual evidence without reflecting the intended activity.\n"
    "- The face crop is not tracked: frames attribute surrounding content to"
    " the detected person, and a center crop would inflate agreement."
    "\n- Stored database times and decoded PTS can disagree; flag such"
    " mismatched metadata instead of silently recentering on it.\n"
    "- Edge clamping repeats boundary frames, which inflates similarity"
    " between the window endpoints.\n"
    "- Softmax scores are relative to the candidate text list, so larger"
    " values do not establish higher quality; an unknown/other candidate is"
    " not a rejection mechanism.\n")
results_table_markdown = (
    "\n".join(summary_lines) + "\n".join(rows_lines)
    + "\n".join(sampling_lines) + notes_text)
display(Markdown(results_table_markdown))

chart_detection = (
    common_successful[0] if common_successful
    else (comparison_rows[0]["detection_id"] if comparison_rows else None))
if chart_detection is not None:
    chart = {
        name: {"retrieval": [], "preprocessing": [], "inference": []}
        for name in CONDITION_NAMES
    }
    for row in comparison_rows:
        if row["detection_id"] != chart_detection:
            continue
        chart[row["condition"]]["retrieval"].append(
            row["retrieval_time_seconds"])
        chart[row["condition"]]["preprocessing"].append(
            row["preprocessing_time_seconds"])
        chart[row["condition"]]["inference"].append(
            row["inference_time_seconds"])
    retrieval = [
        (sum(values) / len(values)) if values else 0.0
        for name in CONDITION_NAMES
        for values in (chart[name]["retrieval"],)]
    preprocessing = [
        (sum(values) / len(values)) if values else 0.0
        for name in CONDITION_NAMES
        for values in (chart[name]["preprocessing"],)]
    inference = [
        (sum(values) / len(values)) if values else 0.0
        for name in CONDITION_NAMES
        for values in (chart[name]["inference"],)]
    figure, axis = plt.subplots(figsize=(8, 4))
    positions = np.arange(len(CONDITION_NAMES))
    axis.bar(positions - 0.25, retrieval, width=0.25, label="retrieval")
    axis.bar(positions, preprocessing, width=0.25, label="preprocessing")
    axis.bar(positions + 0.25, inference, width=0.25, label="inference")
    axis.set_xticks(positions)
    axis.set_xticklabels(CONDITION_NAMES)
    axis.set_ylabel("seconds")
    axis.set_title(f"Per-condition timing for detection {chart_detection}")
    axis.legend()
    display(figure)
    plt.close(figure)
    del figure, axis, chart


In [ ]:
contact_detection = (
    common_successful[0] if common_successful
    else (comparison_rows[0]["detection_id"] if comparison_rows else None))
if contact_detection is not None:
    for name in CONDITION_NAMES:
        row = rows_by_key.get((contact_detection, name))
        if row is None:
            display(Markdown(
                f"- condition {name} has no successful result for detection"
                f" {contact_detection}; it is listed in the exclusions."))
            continue
        top = " · ".join(
            f"{prediction['label']} {prediction['score']:.3f}"
            for prediction in row["predictions"][:TOP_K])
        previews = condition_previews.get((contact_detection, name))
        if previews:
            figure, axes = plt.subplots(1, len(previews), figsize=(9, 3))
            axes = np.ravel(np.array(axes, dtype=object)).tolist()
            for axis, position, image in zip(axes, (0, 16, 31), previews):
                axis.imshow(image)
                axis.set_title(f"slot {position}")
                axis.axis("off")
            figure.suptitle(f"{name}: {top}")
            display(figure)
            plt.close(figure)
            del figure, axes
        else:
            display(Markdown(f"- {name}: {top}"))


## Observations (edit this cell)

- (pending) Compare the rankings across the four conditions using the summary table above.
- Remember the caveats: scene changes and unrelated people in the window are attributed to the detected person; center-crop effects apply; stored-versus-decoded timestamps can disagree; boundary repeats from edge clamping repeat frames; softmax scores are relative to the candidate list, so larger values do not establish higher quality, and an unknown/other candidate is not a rejection mechanism.
